## EDA: Funciones

Lectura del OCDS, capa bronze y aplanado a staging

### Imports

In [ ]:
from pyspark.sql.types import ArrayType, StructType

### **1. Lectura del OCDS**

- Cada archivo mensual es **un** objeto JSON con `records`

- Cada record trae su `compiledRelease` (estado final del proceso)

- `extraer_releases` detecta el envoltorio (`records` / `releases`) en vez de asumirlo

In [ ]:
def listar_archivos(data, anios):
    archivos = []
    for a in anios:
        archivos += sorted((data / str(a)).glob("*.json"))
    return archivos

def leer_json(spark, archivos):
    return spark.read.option("multiLine", "true").option("mode", "PERMISSIVE").json([str(a) for a in archivos])

def extraer_releases(raw):
    top = {c.lower() for c in raw.columns}
    if "records" in top:
        rec = raw.select(F.explode("records").alias("x"))
        return (rec.select("x.compiledRelease.*") if "compiledRelease" in rec.select("x.*").columns
                else rec.select(F.explode("x.releases").alias("r")).select("r.*"))
    if "releases" in top:
        return raw.select(F.explode("releases").alias("r")).select("r.*")
    return raw

def sin_documentos(c):
    # documents es el objeto más numeroso del OCDS y ninguna técnica lo usa
    if c == "tender":
        return F.col("tender").dropFields("documents").alias("tender")
    if c in ("awards", "contracts"):
        return F.transform(c, lambda x: x.dropFields("documents")).alias(c)
    return F.col(c)

### **2. Acceso seguro al esquema**

- El esquema del JSON varía entre meses: un campo ausente devuelve nulo en vez de romper el flujo

- `crear_vista(bronze)` devuelve los accesores `existe`, `col`, `sub` y `explotar` atados a ese esquema

In [ ]:
def rutas(schema, pre=""):
    out = []
    for f in schema.fields:
        out.append(pre + f.name)
        t = f.dataType.elementType if isinstance(f.dataType, ArrayType) else f.dataType
        if isinstance(t, StructType):
            out += rutas(t, pre + f.name + ".")
    return out

def fecha(c):
    return F.to_timestamp(F.substring(c.cast("string"), 1, 19), "yyyy-MM-dd'T'HH:mm:ss")

def crear_vista(bronze):
    rs = {r.lower() for r in rutas(bronze.schema)}
    existe = lambda p: p.lower() in rs

    def col(p, tipo="string"):
        return F.col(p).cast(tipo) if existe(p) else F.lit(None).cast(tipo)

    def sub(alias, padre, campo, tipo="string"):
        # campo de un elemento ya explotado (alias.campo), validando la ruta absoluta padre.campo
        return F.col(f"{alias}.{campo}").cast(tipo) if existe(f"{padre}.{campo}") else F.lit(None).cast(tipo)

    def explotar(ruta, alias):
        if existe(ruta):
            return bronze.select("ocid", F.explode(ruta).alias(alias))
        print(f"aviso: {ruta} no existe en el esquema -> tabla vacía")
        return bronze.select("ocid", F.lit(None).alias(alias)).where("false")

    return existe, col, sub, explotar

### **3. Tablas de staging**

- Una función por tabla

- Cada una devuelve un DataFrame

#### a. Procesos

In [ ]:
def parte_buyer(campo):
    return F.expr(f"element_at(filter(transform(parties, p -> CASE WHEN array_contains(p.roles, 'buyer') "
                  f"THEN {campo} END), x -> x IS NOT NULL), 1)")

def t_procesos(bronze, existe, col):
    ruc = (parte_buyer("element_at(filter(p.additionalIdentifiers, i -> i.scheme = 'PE-RUC'), 1).id")
           if existe("parties.additionalIdentifiers.id") else F.lit(None).cast("string"))
    # en el OCDS del OECE, address.region es la provincia; la región (departamento) está en address.department
    region = parte_buyer("p.address.department") if existe("parties.address.department") else F.lit(None).cast("string")
    provincia = parte_buyer("p.address.region") if existe("parties.address.region") else F.lit(None).cast("string")
    return (bronze.select(
            "ocid",
            F.coalesce(fecha(col("tender.tenderPeriod.startDate")), fecha(F.element_at(col("awards.date", "array<string>"), 1)),
                       fecha(col("date"))).alias("fecha"),
            F.coalesce(col("buyer.id"), col("tender.procuringEntity.id")).alias("entidad_id_raw"),
            ruc.alias("entidad_ruc"),
            F.coalesce(col("buyer.name"), col("tender.procuringEntity.name")).alias("entidad"),
            region.alias("region"),
            provincia.alias("provincia"),
            col("tender.description").alias("descripcion"),
            F.concat_ws(" ", col("tender.items.description", "array<string>")).alias("items_desc"),
            col("tender.procurementMethodDetails").alias("metodo_detalle"),
            col("tender.mainProcurementCategory").alias("categoria"),
            col("tender.value.amount", "double").alias("monto_referencial"),
            col("tender.value.currency").alias("moneda"),
            col("tender.numberOfTenderers", "int").alias("n_postores"),
            F.regexp_extract("_archivo", r"/(\d{4})/(\d{2})\.json$", 1).cast("int").alias("anio_archivo"),
            F.regexp_extract("_archivo", r"/(\d{4})/(\d{2})\.json$", 2).cast("int").alias("mes_archivo"))
        .withColumn("anio", F.coalesce(F.year("fecha"), F.col("anio_archivo"))))

#### b. Adjudicaciones y oferentes

In [ ]:
def t_adjudicaciones(explotar, sub, existe):
    aw = explotar("awards", "a").select(
        "ocid",
        sub("a", "awards", "id").alias("award_id"),
        fecha(sub("a", "awards", "date")).alias("fecha_adj"),
        sub("a", "awards", "status").alias("estado_adj"),
        sub("a", "awards", "value.amount", "double").alias("monto_adj"),
        sub("a", "awards", "value.currency").alias("moneda_adj"),
        sub("a", "awards", "items.classification.id", "array<string>").alias("unspsc_adj"),
        (F.col("a.suppliers") if existe("awards.suppliers")
         else F.lit(None).cast("array<struct<id:string,name:string>>")).alias("sup"))
    # 1 fila por (award, proveedor): explode_outer conserva los awards sin proveedor
    return (aw.withColumn("n_proveedores_adj", F.when(F.col("sup").isNull(), 0).otherwise(F.size("sup")))
              .select("*", F.explode_outer("sup").alias("s")).drop("sup")
              .select("*", sub("s", "awards.suppliers", "id").alias("proveedor_id_raw"),
                           sub("s", "awards.suppliers", "name").alias("proveedor")).drop("s"))

def t_oferentes(explotar, sub):
    return explotar("tender.tenderers", "t").select(
        "ocid", sub("t", "tender.tenderers", "id").alias("oferente_id_raw"),
        sub("t", "tender.tenderers", "name").alias("oferente"))

#### c. Items y contratos

In [ ]:
def t_items(explotar, sub, existe):
    adicional = (F.expr("element_at(filter(coalesce(i.additionalClassifications, array()), "
                        "c -> upper(c.scheme) = 'UNSPSC'), 1).id")
                 if existe("tender.items.additionalClassifications.id") else F.lit(None))
    return explotar("tender.items", "i").select(
        "ocid",
        sub("i", "tender.items", "id").alias("item_id"),
        sub("i", "tender.items", "description").alias("item_desc"),
        sub("i", "tender.items", "quantity", "double").alias("cantidad"),
        F.coalesce(adicional.cast("string"),
                   F.when(F.upper(sub("i", "tender.items", "classification.scheme")) == "UNSPSC",
                          sub("i", "tender.items", "classification.id"))).alias("unspsc"))

def t_contratos(explotar, sub):
    return explotar("contracts", "c").select(
        "ocid",
        sub("c", "contracts", "id").alias("contrato_id"),
        sub("c", "contracts", "awardID").alias("award_id"),
        sub("c", "contracts", "status").alias("estado_contrato"),
        sub("c", "contracts", "value.amount", "double").alias("monto_contrato"),
        sub("c", "contracts", "value.currency").alias("moneda_contrato"),
        fecha(sub("c", "contracts", "dateSigned")).alias("fecha_firma"))

def t_partes(explotar, sub):
    return explotar("parties", "p").select(
        "ocid",
        sub("p", "parties", "id").alias("parte_id_raw"),
        sub("p", "parties", "name").alias("nombre"),
        sub("p", "parties", "identifier.id").alias("identificador"),
        sub("p", "parties", "identifier.scheme").alias("esquema"),
        sub("p", "parties", "roles", "array<string>").alias("roles"),
        sub("p", "parties", "address.department").alias("region"),
        sub("p", "parties", "address.region").alias("provincia"))

### **4. Formato de identificadores**

- RUC = 11 dígitos (`10…` persona natural, `20…` persona jurídica); DNI = 8 dígitos

- `PE-CONSUCODE-<6 dígitos>`: código de entidad compradora en el SEACE

- `PE-RUC-<6-7 dígitos>`: no es un RUC; es el código que el SEACE asigna a un **consorcio**

- `PE-RUC-L<10 dígitos>`: proveedor extranjero (no domiciliado) sin RUC

In [ ]:
def patron_id(c):
    return (F.when(F.col(c).isNull(), "nulo")
             .when(F.col(c).rlike(r"^PE-CONSUCODE-\d+$"), "CONSUCODE (entidad)")
             .when(F.col(c).rlike(r"RUC\D?\d{11}$"), "RUC 11 dígitos")
             .when(F.col(c).rlike(r"^PE-RUC-\d{6,7}$"), "código de consorcio")
             .when(F.col(c).rlike(r"^PE-RUC-L\d{10}$"), "extranjero (L + 10 dígitos)")
             .when(F.col(c).rlike(r"(?<!\d)\d{8}$"), "DNI 8 dígitos")
             .otherwise("otro"))